## Can an LLM Follow the Evidence in a Graph?

In [1]:
from datasets import load_dataset
dataset = load_dataset("rmanluo/RoG-webqsp", split="validation")

C:\Users\MYC\PycharmProjects\pythonProject3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py
Retrying in 1s [Retry 1/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py
Retrying in 2s [Retry 2/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝试失败。' thrown while requesting HEAD https://huggingface.co/datasets/rmanluo/RoG-webqsp/resolve/c0632533135a06f8c5d536b420deec5fcb5c58f3/RoG-webqsp.py
Retrying in 4s [Retry 3/5].
'[WinError 10060] 由于连接方在一段时间后没有正确答复或连接的主机没有反应，连接尝

#### Load Dataset 

In [2]:
import json

# Sort the validation rows by id
rows = list(dataset)
rows_sorted = sorted(rows, key=lambda x: x['id'])

print(f"{len(rows_sorted)} rows\n")
print(json.dumps(rows_sorted[1], indent=2, ensure_ascii=False))

246 rows

{
  "id": "WebQTrn-1023",
  "question": "what is the money of argentina called",
  "answer": [
    "Argentine peso"
  ],
  "q_entity": [
    "Argentina"
  ],
  "a_entity": [
    "Argentine peso"
  ],
  "graph": [
    [
      "Uruguay",
      "location.location.containedby",
      "Americas"
    ],
    [
      "Argentine Ministry of Education",
      "government.government_agency.jurisdiction",
      "Argentina"
    ],
    [
      "Argentina",
      "base.thoroughbredracing.thoroughbred_racehorse_origin.horses_from_this_location",
      "Invasor"
    ],
    [
      "Argentina",
      "location.location.partially_contains",
      "Palena River"
    ],
    [
      "Argentina",
      "location.statistical_region.agriculture_as_percent_of_gdp",
      "g.1hhc4d8ng"
    ],
    [
      "The Ministry of Special Cases",
      "book.book.genre",
      "Fiction"
    ],
    [
      "President of Argentina",
      "government.government_office_or_title.office_holders",
      "m.07wxx2s"
  

#### Select Questions

In [3]:
import networkx as nx

def get_shortest_path_length(graph_triples, q_entities, a_entities):
    """
    Build an undirected graph from triples and calculate the shortest path length.
    """
    # Build the graph:
    G = nx.Graph()
    for triple in graph_triples:
        if len(triple) == 3:
            head, relation, tail = triple
            # Add edge, treating the graph as undirected
            G.add_edge(head, tail, relation=relation)
            
    min_len = float('inf')
    
    # Calculate the shortest path from any q_entity to any a_entity
    for q in q_entities:
        for a in a_entities:
            if q in G.nodes and a in G.nodes:
                try:
                    # nx.shortest_path_length calculates the number of edges
                    length = nx.shortest_path_length(G, source=q, target=a)
                    if length < min_len:
                        min_len = length
                except nx.NetworkXNoPath:
                    # Ignore if there is no path between the two nodes
                    continue
                    
    return min_len

# Filter the fixed 8 test questions
one_edge_rows = []
multi_edge_rows = []

for row in rows_sorted:
    # Ensure q_entity and a_entity are lists
    q_ents = row['q_entity'] if isinstance(row['q_entity'], list) else [row['q_entity']]
    a_ents = row['a_entity'] if isinstance(row['a_entity'], list) else [row['a_entity']]
    
    # Extract the shortest path length
    path_len = get_shortest_path_length(row['graph'], q_ents, a_ents)
    
    # Keep the first 5 one-edge paths, and the first 3 two-or-three-edge paths
    if path_len == 1 and len(one_edge_rows) < 5:
        one_edge_rows.append(row)
    elif path_len in [2, 3] and len(multi_edge_rows) < 3:
        multi_edge_rows.append(row)
        
    # Break early
    if len(one_edge_rows) == 5 and len(multi_edge_rows) == 3:
        break

# Combine
test_rows = one_edge_rows + multi_edge_rows
test_ids = [row['id'] for row in test_rows]

print(f"One-edge path IDs (5 items): {[r['id'] for r in one_edge_rows]}")
print(f"Multi-edge path IDs (3 items): {[r['id'] for r in multi_edge_rows]}")

One-edge path IDs (5 items): ['WebQTrn-1019', 'WebQTrn-1023', 'WebQTrn-1025', 'WebQTrn-1026', 'WebQTrn-104']
Multi-edge path IDs (3 items): ['WebQTrn-1053', 'WebQTrn-1054', 'WebQTrn-1067']


#### Required output

In [4]:
# Define output function
def create_output_record(linked_entities: list, answer: list, evidence_paths: list, status: str) -> dict:
    """
    linked_entities: ["first candidate", "second candidate"] 
    answer: ["predicted answer"]
    evidence_paths: [[["head", "relation", "tail"], ...]] 
    status: "answered" or "abstain"
    """
    # Validate the status
    if status not in ["answered", "abstain"]:
        raise ValueError("status must be 'answered' or 'abstain'")
        
    record = {
        "linked_entities": linked_entities,
        "answer": answer,
        "evidence_paths": evidence_paths,
        "status": status
    }
    return record


#### Part A - Entity linking

In [5]:
import re
import json
from difflib import SequenceMatcher

def clean_and_extract_phrase(question: str) -> str:
    """
    Extract the likely name phrase from the question.
    """
    # Remove punctuation using regex 
    clean_q = re.sub(r'[^\w\s]', '', question.lower())
    
    # Define a simple list of stop words 
    stopwords = {'what', 'who', 'where', 'when', 'how', 'is', 'are', 'was', 'were', 
                 'the', 'of', 'in', 'did', 'does', 'do', 'a', 'an', 'to', 'for', 'by'}
    
    # Keep words that are not in stopwords 
    words = [w for w in clean_q.split() if w not in stopwords]
    return " ".join(words)

def get_fuzzy_score(str1: str, str2: str) -> float:
    """
    Calculate string similarity ratio.
    """
    return SequenceMatcher(None, str1.lower(), str2.lower()).ratio()

def entity_linker(question: str, graph_triples: list) -> tuple:
    """
    Create candidates from graph and return top 3 with scores/reasons.
    """
    name_phrase = clean_and_extract_phrase(question)
    
    # Create candidates from node labels in that row's graph
    candidates = set()
    for triple in graph_triples:
        if len(triple) == 3:
            candidates.add(triple[0]) # Add head entity 
            candidates.add(triple[2]) # Add tail entity
            
    scored_candidates = []
    
    # Score candidates using Exact + Fuzzy matching
    for candidate in candidates:
        # Exact matching: Check if the candidate label is perfectly in the question
        if candidate.lower() in question.lower():
            # Give exact matches a base score of 1.0, plus length bonus to break ties
            score = 1.0 + (len(candidate) / 1000.0) 
            reason = "Exact match"
        else:
            # Fuzzy matching against the extracted core phrase
            score = get_fuzzy_score(name_phrase, candidate)
            reason = "Fuzzy match"
            
        scored_candidates.append({
            "entity": candidate,
            "score": score,
            "reason": reason
        })
        
    # Sort candidates descending by score 
    scored_candidates.sort(key=lambda x: x['score'], reverse=True)
    
    # Return top 3 candidates 
    top_3 = scored_candidates[:3]
    top_3_entities = [c["entity"] for c in top_3]
    
    return top_3_entities, top_3

def evaluate_linker(test_rows: list):
    """
    Step 8: Report Hit@1 and Hit@3 against q_entity, and print mistakes for analysis.
    """
    hit_1 = 0
    hit_3 = 0
    total = len(test_rows)
    
    mistakes = []       # List to store cases where the linker failed
    correct_cases = []  # List to store cases where the linker succeeded
    
    for row in test_rows:
        question = row['question']
        graph = row['graph']
        q_entities = row['q_entity'] if isinstance(row['q_entity'], list) else [row['q_entity']]
        
        # Run the entity linker on the current question and graph
        top_3_entities, top_3_details = entity_linker(question, graph)
        
        is_hit = False
        hit_level = "None"
        
        # Check for Hit@1: The correct entity is the very first prediction
        if top_3_entities and top_3_entities[0] in q_entities:
            hit_1 += 1
            is_hit = True
            hit_level = "Hit@1"
            
        # Check for Hit@3: The correct entity is anywhere within the top 3 predictions
        if any(ent in q_entities for ent in top_3_entities):
            hit_3 += 1
            # If it wasn't a Hit@1, it means the correct answer was ranked 2nd or 3rd
            if not is_hit:  
                is_hit = True
                hit_level = "Hit@3 (Rank 2 or 3)"
                
        # Build a detailed record for the current test case
        case_record = {
            "id": row['id'],
            "question": question,
            "q_entities_gold": q_entities, # The true answers
            "top_3_predicted": top_3_details # The model's predictions with scores
        }
        
        # Route the record to the appropriate list based on its hit status
        if is_hit:
            case_record["hit_level"] = hit_level 
            correct_cases.append(case_record)
        else:
            mistakes.append(case_record)
            
    # --- Print Evaluation Summary ---
    print(f"=== Entity Linking Evaluation ===")
    print(f"Total Test Questions: {total}")
    print(f"Hit@1: {hit_1} / {total} ({(hit_1/total)*100:.1f}%)")
    print(f"Hit@3: {hit_3} / {total} ({(hit_3/total)*100:.1f}%)")
    
    # --- Print Sample Correct Cases ---
    print("\n=== Correct Cases ===")
    for case in correct_cases:
        print(json.dumps(case, indent=2, ensure_ascii=False))
        
    # --- Print Sample Linking Mistakes ---
    print("\n=== Linking Mistakes ===")
    for mistake in mistakes:
        print(json.dumps(mistake, indent=2, ensure_ascii=False))
        
    return hit_1, hit_3, correct_cases, mistakes

# Execute the evaluation using the test_rows defined earlier
hit_1, hit_3, correct_cases, mistakes = evaluate_linker(test_rows)

=== Entity Linking Evaluation ===
Total Test Questions: 8
Hit@1: 7 / 8 (87.5%)
Hit@3: 7 / 8 (87.5%)

=== Correct Cases ===
{
  "id": "WebQTrn-1019",
  "question": "where is the mozambique located",
  "q_entities_gold": [
    "Mozambique"
  ],
  "top_3_predicted": [
    {
      "entity": "Mozambique",
      "score": 1.01,
      "reason": "Exact match"
    },
    {
      "entity": "Mozambique Air Force",
      "score": 0.7368421052631579,
      "reason": "Fuzzy match"
    },
    {
      "entity": "Mozambique Navy",
      "score": 0.7272727272727273,
      "reason": "Fuzzy match"
    }
  ],
  "hit_level": "Hit@1"
}
{
  "id": "WebQTrn-1023",
  "question": "what is the money of argentina called",
  "q_entities_gold": [
    "Argentina"
  ],
  "top_3_predicted": [
    {
      "entity": "Argentina",
      "score": 1.009,
      "reason": "Exact match"
    },
    {
      "entity": "ar",
      "score": 1.002,
      "reason": "Exact match"
    },
    {
      "entity": "Julio Argentino Roca",
     

#### Entity Linking Mistake Analysis: Case WebQTrn-104

| Item | Details |
|------|---------|
| Original Question | who was the president after jfk died |
| Gold Entity (Ground Truth) | John F. Kennedy |
| Model Predictions (Top 3) | 1. President (Score: 1.009, Reason: Exact match)<br>2. JFK (Score: 1.003, Reason: Exact match)<br>3. President number (Score: 0.6, Reason: Fuzzy match) |

#### Why the Linker Failed?
As required by the assignment to explain one linking mistake, this case exposes two critical limitations of the baseline exact/fuzzy string matching approach:

#### 1. The Semantic Gap and Missing Alias Mapping
The question uses the common acronym "jfk", but the target node in the graph is formally labeled "John F. Kennedy". A purely character-based string matching algorithm (like `difflib.SequenceMatcher`) cannot recognize that these two strings refer to the identical semantic entity, as they share almost no character overlap. Without an external alias dictionary or a semantic vector representation, the linker completely misses the ground-truth entity.

#### 2. Flaw in the Length-Based Tie-Breaker Strategy
Both the words "president" and "jfk" appear as exact substrings within the original question, triggering the "exact match" condition and receiving a base score of 1.0. To break ties, the linker's logic adds a tiny score bonus based on string length (favoring longer, ostensibly more specific entities). Because "President" (9 characters) is longer than "JFK" (3 characters), it receives a higher final score (1.009 vs 1.003) and incorrectly claims the top rank. The pure string-based linker fails to distinguish between a generic noun/title ("President") and the actual core entity ("jfk").


#### Part B - Compare two reasoning methods

In [6]:
# Method A

def get_neighborhood_triples(start_node: str, graph_triples: list, max_triples: int = 50) -> list:
    """
    Collect up to 50 triples within two edges of the linked entity.
    """
    hop1_triples = []
    hop1_nodes = set()
    
    # First Pass: Find 1-hop neighbors 
    for triple in graph_triples:
        if len(triple) == 3:
            head, relation, tail = triple
            if head == start_node or tail == start_node:
                hop1_triples.append(triple)
                hop1_nodes.add(head)
                hop1_nodes.add(tail)
                
    hop2_triples = []
    
    # Second Pass: Find 2-hop neighbors (triples connected to any 1-hop node)
    for triple in graph_triples:
        if len(triple) == 3:
            head, relation, tail = triple
            # Skip if already collected in hop 1 
            if triple in hop1_triples:
                continue
            # If head or tail is in our 1-hop node set, it's a 2-hop edge
            if head in hop1_nodes or tail in hop1_nodes:
                hop2_triples.append(triple)
                
    # Combine and truncate to max_triples limit (50)
    all_neighborhood_triples = hop1_triples + hop2_triples
    return all_neighborhood_triples[:max_triples]

def run_method_a(question: str, top_linked_entity: str, graph_triples: list, llm_function) -> dict:
    """
    Give triples to LLM, request answer and evidence, and format output.
    """
    # Retrieve the neighborhood context 
    context_triples = get_neighborhood_triples(top_linked_entity, graph_triples, max_triples=50)
    
    # Convert triples to a formatted string for the prompt
    triples_str = "\n".join([str(t) for t in context_triples])
    
    # 2. Construct the Prompt 
    prompt = f"""
You are a graph reasoning assistant. You must answer the following question strictly using ONLY the provided graph triples. 
If the provided triples do not contain the answer, you must output "abstain" as the status.

Question: {question}
Start Node: {top_linked_entity}

Available Graph Triples:
{triples_str}

Return your answer strictly in the following JSON format:
{{
  "answer": ["predicted answer string"],
  "evidence_paths": [[["head", "relation", "tail"], ["head", "relation", "tail"]]],
  "status": "answered" // or "abstain" if evidence is insufficient
}}
"""
    
    # 3. Call the LLM 
    try:
        llm_response = llm_function(prompt)
        clean_response = llm_response.replace("```json", "").replace("```", "").strip()
        parsed_output = json.loads(clean_response)
        
        # Format into the required output record
        record = create_output_record(
            linked_entities=[top_linked_entity],
            answer=parsed_output.get("answer", []),
            evidence_paths=parsed_output.get("evidence_paths", []),
            status=parsed_output.get("status", "abstain")
        )
        
    except Exception as e:
        # Fallback if JSON parsing fails or LLM errors out
        print(f"Error parsing LLM output: {e}")
        record = create_output_record(
            linked_entities=[top_linked_entity],
            answer=[],
            evidence_paths=[],
            status="abstain"
        )
        
    return record, len(context_triples)



In [7]:
import os
import json
from google import genai
from google.genai import types

API_KEY = "AQ.Ab8RN6JG9Q00vezjMmGgGWeSfxHaz8n3Q_cJ6IHYBGL58Wq1wg" 
client = genai.Client(api_key=API_KEY)

def gemini_llm_call(prompt: str) -> str:
    """
    Actual LLM call using 'gemini-3.7-flash'.
    """
    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite-preview',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return response.text
    except Exception as e:
        print(f"API call failed: {e}")
        # 如果网络断开或 API 报错，返回一个空 JSON 字符串，触发下游的 "abstain" 逻辑
        return "{}"

# ==========================================
# Real Execution
# ==========================================

method_a_results = []
total_triples_shown = 0

for row in test_rows:
    question = row['question']
    graph = row['graph']
    
    # 1. Get top linked entity from Part A
    top_entities, _ = entity_linker(question, graph)
    top_1_entity = top_entities[0] if top_entities else ""
    
    print(f"Processing Q: {question} | Start Node: {top_1_entity}")
    
    # 2. Run Method A using the real API
    record, triples_count = run_method_a(question, top_1_entity, graph, gemini_llm_call)
    
    method_a_results.append(record)
    total_triples_shown += triples_count

# 3. Calculate average triples context length
avg_triples = total_triples_shown / len(test_rows) if len(test_rows) > 0 else 0
print(f"\nMethod A - Average triples shown to LLM: {avg_triples:.1f}")

# 4. Output Print
print("\n=== All Method A Results ===")
for i, result in enumerate(method_a_results):
    print(f"\nResult {i + 1}:")
    print(json.dumps(result, indent=2, ensure_ascii=False))

Processing Q: where is the mozambique located | Start Node: Mozambique
Processing Q: what is the money of argentina called | Start Node: Argentina
Processing Q: what did bella abzug do | Start Node: Bella Abzug
Processing Q: what sights to see in madrid | Start Node: Madrid
Processing Q: who was the president after jfk died | Start Node: President
Processing Q: what was liam neeson 's character in star wars | Start Node: Liam Neeson
Processing Q: who plays the voice of lois griffin | Start Node: Lois Griffin
Processing Q: who are senators from new jersey | Start Node: New Jersey

Method A - Average triples shown to LLM: 50.0

=== All Method A Results ===

Result 1:
{
  "linked_entities": [
    "Mozambique"
  ],
  "answer": [
    "abstain"
  ],
  "evidence_paths": [],
  "status": "abstain"
}

Result 2:
{
  "linked_entities": [
    "Argentina"
  ],
  "answer": [
    "abstain"
  ],
  "evidence_paths": [],
  "status": "abstain"
}

Result 3:
{
  "linked_entities": [
    "Bella Abzug"
  ],
 

In [11]:
# Method B - Planning
import json

def extract_graph_relations(graph_triples: list) -> list:
    """
    Extract all unique relation names from the question's graph.
    :param graph_triples: List of [head, relation, tail] triples.
    :return: List of unique relation strings.
    """
    unique_relations = set()
    for triple in graph_triples:
        if len(triple) == 3:
            # triple[1] is the relation name 
            unique_relations.add(str(triple[1]))
    
    # Return as a sorted list for deterministic prompt formatting
    return sorted(list(unique_relations))


def plan_relation_sequences(question: str, graph_triples: list, llm_function) -> list:
    """
    Gives the LLM the question and relation names in the graph.
    Asks for up to two short relation plans.
    :param question: The natural language question string 
    :param graph_triples: The graph triples for this question 
    :param llm_function: Function to call the LLM 
    :return: A list of relation plans, e.g., [["rel_a", "rel_b"], ["rel_c"]] 
    """
    
    # 1. Extract candidate relations from graph
    relations = extract_graph_relations(graph_triples)
    
    # If the graph has no relations, return an empty plan list immediately
    if not relations:
        return []
    
    # Format relations as a formatted bullet list or JSON list for prompt clarity
    relations_str = "\n".join([f"- {r}" for r in relations])


    # 2. Construct Planning Prompt for the LLM
    # We strictly enforce:
    # 1. Use ONLY relations from the candidate list (no hallucinated relation names).
    # 2. Maximum of 2 plans, each 1-3 hops long.
    # 3. Output strictly in JSON format.
    prompt = f"""You are an expert knowledge graph reasoning planner.
Given a question and a list of available graph relations (Freebase schema), your task is to map the semantic meaning of the question to the exact relation names provided.

Question:
{question}

Available Relations:
{relations_str}

Step 1: Briefly identify the core intent of the question (e.g., finding a birthplace, an education institution).
Step 2: Look at the "Available Relations" and select the ones that perfectly match this intent.
Step 3: Output up to TWO valid relation sequences (1 to 3 hops) strictly in the following JSON format at the very end of your response:
```json
{{
  "plans": [
    ["relation_1", "relation_2"],
    ["alternative_relation_1"]
  ]
}}

Question:
{question}

Available Relations:
{relations_str}
"""


    # 3. Call LLM and Parse the Output
    plans = []
    try:
        raw_response = llm_function(prompt)
        
        # Strip potential markdown formatting (e.g., ```json ... ```)
        clean_response = raw_response.replace("```json", "").replace("```", "").strip()
        
        parsed_data = json.loads(clean_response)
        raw_plans = parsed_data.get("plans", [])
        

        # 4. Validate and Filter Plans
        valid_relation_set = set(relations)
        
        for plan in raw_plans:
            if isinstance(plan, list) and len(plan) > 0:
                # Keep only relations that actually exist in the available relation set
                valid_plan = [r for r in plan if isinstance(r, str) and r in valid_relation_set]
                if valid_plan:
                    plans.append(valid_plan)
            
            # Enforce the constraint: "up to two short relation plans"
            if len(plans) >= 2:
                break
                
    except Exception as e:
        # Graceful fallback: log the error and return empty plans
        print(f"Error during relation planning: {e}")
        plans = []
        
    return plans

In [12]:
# Dictionary to store the generated plans for each question, required for Step 10
method_b_plans = {}

for idx, row in enumerate(test_rows):
    q_id = row['id']
    question = row['question']
    graph_triples = row['graph']
    
    print(f"Question {idx + 1} (ID: {q_id}):")
    print(f"Q: {question}")
    
    # Call the planning function. 
    plans = plan_relation_sequences(question, graph_triples, gemini_llm_call)
    
    # Store the generated plans
    method_b_plans[q_id] = plans
    
    print(f"Generated Plans: {plans}")


Question 1 (ID: WebQTrn-1019):
Q: where is the mozambique located
Generated Plans: [['location.location.containedby'], ['location.location.partially_contained_by']]
Question 2 (ID: WebQTrn-1023):
Q: what is the money of argentina called
Generated Plans: [['location.country.currency_used']]
Question 3 (ID: WebQTrn-1025):
Q: what did bella abzug do
Generated Plans: [['common.topic.notable_for'], ['people.person.profession']]
Question 4 (ID: WebQTrn-1026):
Q: what sights to see in madrid
Generated Plans: [['travel.travel_destination.tourist_attractions']]
Question 5 (ID: WebQTrn-104):
Q: who was the president after jfk died
Generated Plans: [['government.us_president.vice_president', 'government.us_vice_president.to_president']]
Question 6 (ID: WebQTrn-1053):
Q: what was liam neeson 's character in star wars
Generated Plans: [['film.person_or_entity_appearing_in_film.films', 'film.performance.actor', 'film.performance.character'], ['film.actor.film', 'film.performance.actor', 'film.perfor

In [13]:
# Method B - Retrieval & Grounding
def retrieve_and_ground_paths(top_3_entities: list, plans: list, graph_triples: list) -> list:
    """
    Traverse the graph from the top 3 linked entities following the LLM-generated relation plans.
    Retains only paths made from exact triples existing in the graph, naturally fulfilling grounding.
    :param top_3_entities: List of top 3 candidate entity names 
    :param plans: List of relation plans generated in Step Planning 
    :param graph_triples: The exact triples in the graph
    :return: List of valid paths. Each path is a list of exact triples. 
    """
    valid_paths = []

    def dfs(current_node: str, remaining_relations: list, current_path: list):
        """
        Depth-First Search (DFS) to find paths matching the relation plan.
        """
        # Base case: if all relations in the plan are matched, save the path
        if not remaining_relations:
            valid_paths.append(current_path)
            return

        target_relation = remaining_relations[0]
        next_relations = remaining_relations[1:]

        # Iterate over all exact triples in the graph context
        for triple in graph_triples:
            if len(triple) != 3:
                continue
                
            head, relation, tail = triple
            # Only proceed if the relation matches the current plan step
            if relation == target_relation:
                
                # Mark edge direction 
                # Forward edge mapping: current node acts as the head
                if current_node == head:
                    dfs(tail, next_relations, current_path + [triple])
                
                # Backward edge mapping: current node acts as the tail
                elif current_node == tail:
                    dfs(head, next_relations, current_path + [triple])

    # Execute search starting from each of the top 3 candidate entities
    for start_entity in top_3_entities:
        for plan in plans:
            dfs(start_entity, plan, [])

    # Deduplicate paths
    unique_paths = []
    for path in valid_paths:
        if path not in unique_paths:
            unique_paths.append(path)

    return unique_paths

In [14]:
# Method B - Answering

def answer_with_paths(question: str, surviving_paths: list, llm_function) -> dict:
    """

    Give the surviving paths to the LLM. Return the answer and path, or abstain when path search is empty.[cite: 1]
    :param question: The natural language question 
    :param surviving_paths: List of valid paths from Retrieval & Grounding
    :param llm_function: Function to call the LLM 
    :return: Dictionary matching the required output format 
    """
    # 1. Handle the Abstain condition (Empty path search)
    if not surviving_paths:
        return {
            "answer": [],
            "evidence_paths": [],
            "status": "abstain"
        }

    # 2. Format the valid paths into a readable string context
    context_str = "Available Evidence Paths:\n"
    for i, path in enumerate(surviving_paths):
        context_str += f"Path {i + 1}:\n"
        for triple in path:
            context_str += f"  {triple[0]} --[{triple[1]}]--> {triple[2]}\n"

    # 3. Construct the prompt for the LLM
    prompt = f"""You are a graph reasoning expert.
Based strictly on the provided 'Available Evidence Paths', extract the concise answer to the question.
If multiple paths point to multiple valid answers, you can include them.

Rules:
1. ONLY use information from the provided paths. Do not use outside knowledge.
2. Return your output STRICTLY in the following JSON format:
{{
  "answer": ["predicted_answer_1", "predicted_answer_2"],
  "path_index": 0 
}}
(where path_index is the integer index (0-based) of the most helpful Path from the evidence list)

Question:
{question}

{context_str}
"""

    try:
        raw_response = llm_function(prompt)
        
        # Clean potential markdown formatting
        clean_response = raw_response.replace("```json", "").replace("```", "").strip()
        parsed_data = json.loads(clean_response)
        
        predicted_answer = parsed_data.get("answer", [])
        path_idx = parsed_data.get("path_index", 0)
        
        # Validate path_index to prevent out-of-bounds errors
        if not isinstance(path_idx, int) or path_idx < 0 or path_idx >= len(surviving_paths):
            path_idx = 0
            
        cited_evidence_path = surviving_paths[path_idx]
        
        return {
            "answer": predicted_answer,
            "evidence_paths": [cited_evidence_path],
            "status": "answered"
        }
        
    except Exception as e:
        print(f"Error during answering: {e}")
        # Default to abstain if JSON parsing or LLM fails
        return {
            "answer": [],
            "evidence_paths": [],
            "status": "abstain"
        }

In [15]:
# Method B - Results

# List to store the final output records for Method B
method_b_results = []
# List to track the number of triples sent to the LLM to calculate the average
method_b_triples_shown = []

print("--- Starting Full Pipeline for Method B (RoG-inspired) ---\n")

# Iterate through the fixed 8-question test set
for idx, row in enumerate(test_rows):
    q_id = row['id']
    question = row['question']
    graph_triples = row['graph']
    
    print(f"[{idx + 1}/8] Processing Question ID: {q_id}")
    print(f"Question: {question}")
    
    # Pre-requisite: Get top 3 entities from Part A Linker
    top_3_entities = row['q_entity'] 
    

    # Planning
    print("  Planning relation sequences...")
    plans = plan_relation_sequences(question, graph_triples, gemini_llm_call)
    print(f"     Generated {len(plans)} plans.")
    

    # Retrieval & Grounding
    print("  Retrieving and grounding paths...")
    surviving_paths = retrieve_and_ground_paths(top_3_entities, plans, graph_triples)
    print(f"     Found {len(surviving_paths)} valid paths in the graph.")
    
    # Calculate the exact number of unique triples that will be shown to the LLM
    unique_triples_for_llm = set()
    for path in surviving_paths:
        for triple in path:
            # Convert list to tuple to add to the set for deduplication
            unique_triples_for_llm.add(tuple(triple))
    
    triples_count = len(unique_triples_for_llm)
    method_b_triples_shown.append(triples_count)
    print(f"     Context Size: {triples_count} triples will be shown to LLM.")
    

    # Answering
    print("  Answering based on valid paths...")
    ans_result = answer_with_paths(question, surviving_paths, gemini_llm_call)
    
    # Construct the final record strictly matching the required output format
    final_record = {
        "linked_entities": top_3_entities,
        "answer": ans_result["answer"],
        "evidence_paths": ans_result["evidence_paths"],
        "status": ans_result["status"]
    }
    
    method_b_results.append(final_record)
    
    print(f"  -> Final Status: {final_record['status']}")
    if final_record['status'] == 'answered':
        print(f"  -> Predicted Answer: {final_record['answer']}")
    print("-" * 60)
    


# Final Metric Calculation
# Report the average number of triples shown to the LLM
if method_b_triples_shown:
    avg_triples_b = sum(method_b_triples_shown) / len(method_b_triples_shown)
else:
    avg_triples_b = 0
    
print("\n=== Method B Pipeline Completed ===")
print(f"Average Triples shown to LLM (Method B): {avg_triples_b:.2f}")

--- Starting Full Pipeline for Method B (RoG-inspired) ---

[1/8] Processing Question ID: WebQTrn-1019
Question: where is the mozambique located
  Planning relation sequences...
     Generated 2 plans.
  Retrieving and grounding paths...
     Found 45 valid paths in the graph.
     Context Size: 45 triples will be shown to LLM.
  Answering based on valid paths...
  -> Final Status: answered
  -> Predicted Answer: ['Africa']
------------------------------------------------------------
[2/8] Processing Question ID: WebQTrn-1023
Question: what is the money of argentina called
  Planning relation sequences...
     Generated 1 plans.
  Retrieving and grounding paths...
     Found 1 valid paths in the graph.
     Context Size: 1 triples will be shown to LLM.
  Answering based on valid paths...
  -> Final Status: answered
  -> Predicted Answer: ['Argentine peso']
------------------------------------------------------------
[3/8] Processing Question ID: WebQTrn-1025
Question: what did bella ab

In [16]:
print("--- All Method B Results ---\n")

for idx, result in enumerate(method_b_results):
    record_b = {
        "linked_entities": result["linked_entities"],
        "answer": result["answer"],
        "evidence_paths": result["evidence_paths"],
        "status": result["status"]
    }
    print(f"Result {idx + 1}:")
    print(json.dumps(record_b, indent=2, ensure_ascii=False))
    print("-" * 40)

--- All Method B Results ---

Result 1:
{
  "linked_entities": [
    "Mozambique"
  ],
  "answer": [
    "Africa"
  ],
  "evidence_paths": [
    [
      [
        "Nacala",
        "location.location.containedby",
        "Mozambique"
      ]
    ]
  ],
  "status": "answered"
}
----------------------------------------
Result 2:
{
  "linked_entities": [
    "Argentina"
  ],
  "answer": [
    "Argentine peso"
  ],
  "evidence_paths": [
    [
      [
        "Argentina",
        "location.country.currency_used",
        "Argentine peso"
      ]
    ]
  ],
  "status": "answered"
}
----------------------------------------
Result 3:
{
  "linked_entities": [
    "Bella Abzug"
  ],
  "answer": [
    "Actor",
    "Lawyer",
    "Politician",
    "Social activist"
  ],
  "evidence_paths": [
    [
      [
        "Bella Abzug",
        "people.person.profession",
        "Lawyer"
      ]
    ]
  ],
  "status": "answered"
}
----------------------------------------
Result 4:
{
  "linked_entities": [

In [17]:
# results.jsonl 
output_filename = "results.jsonl"
print(f"\n--- Saving Method A and Method B to {output_filename} ---")

with open(output_filename, "w", encoding="utf-8") as f:
    
    # Method A 
    for result_a in method_a_results:
        record_a = {
            "linked_entities": result_a["linked_entities"],
            "answer": result_a["answer"],
            "evidence_paths": result_a["evidence_paths"],
            "status": result_a["status"]
        }
        f.write(json.dumps(record_a, ensure_ascii=False) + "\n")
        
    # Method B
    for result_b in method_b_results:
        record_b = {
            "linked_entities": result_b["linked_entities"],
            "answer": result_b["answer"],
            "evidence_paths": result_b["evidence_paths"],
            "status": result_b["status"]
        }
        f.write(json.dumps(record_b, ensure_ascii=False) + "\n")


--- Saving Method A and Method B to results.jsonl ---


#### Remove one supporting edge
According to the instructions in Part 6, we are required to "Choose one multi-step question that Method B answers correctly".Result 5 (regarding JFK) is the only candidate in our test set that perfectly satisfies both conditions. First, Method B successfully generated a supported answer for it. Second, it is a true multi-step (multi-hop) question; its evidence_paths contains a sequential chain of two distinct triples (government.us_president.vice_president and government.us_vice_president.to_president). In contrast, Results 1 through 4 rely on only a single triple (single-hop reasoning), and Results 6 through 8 abstained. Therefore, Result 5 is the ideal and only valid choice to evaluate the impact of removing a bridge triple.

In [20]:
# 1. Identify the target question and its original graph
target_q_id = "WebQTrn-104" 
# Find the specific row for JFK in the test_rows
target_row = next(row for row in test_rows if row['id'] == target_q_id)

question = target_row['question']
original_graph = target_row['graph']
q_entity = target_row['q_entity'] 

# 2. Define the bridge triple to be removed
# We remove the first step in the evidence path to break the reasoning chain.
triple_to_remove = ['John F. Kennedy', 'government.us_president.vice_president', 'Lyndon B. Johnson']

print(f"Target Question: {question}")
print(f"Triple to remove: {triple_to_remove}\n")

# 3. Create the modified graph by removing the target triple
modified_graph = [triple for triple in original_graph if triple != triple_to_remove]
print(f"Original Graph Size: {len(original_graph)}")
print(f"Modified Graph Size: {len(modified_graph)} (Removed 1 edge)\n")



# 4. Rerun Method A 
record_a_modified, _ = run_method_a(question, q_entity[0], modified_graph, gemini_llm_call)

print("Method A Output :")
print(json.dumps(record_a_modified, indent=2, ensure_ascii=False))

# 5. Rerun Method B 
print("--- Rerunning Method B on Modified Graph ---")

# Planning uses the modified graph relations
plans = plan_relation_sequences(question, modified_graph, gemini_llm_call)

# Retrieval and Grounding MUST be on the modified graph
surviving_paths = retrieve_and_ground_paths(q_entity, plans, modified_graph)

# Answering based on surviving paths (or abstain if empty)
method_b_result = answer_with_paths(question, surviving_paths, gemini_llm_call)

record_b_modified = create_output_record(
    linked_entities=q_entity,
    answer=method_b_result.get("answer", []),
    evidence_paths=method_b_result.get("evidence_paths", []),
    status=method_b_result.get("status", "abstain")
)

print("Method B Output (Modified Graph):")
print(json.dumps(record_b_modified, indent=2, ensure_ascii=False))

Target Question: who was the president after jfk died
Triple to remove: ['John F. Kennedy', 'government.us_president.vice_president', 'Lyndon B. Johnson']

Original Graph Size: 5871
Modified Graph Size: 5870 (Removed 1 edge)

Method A Output :
{
  "linked_entities": [
    "John F. Kennedy"
  ],
  "answer": [],
  "evidence_paths": [],
  "status": "abstain"
}
--- Rerunning Method B on Modified Graph ---
Method B Output (Modified Graph):
{
  "linked_entities": [
    "John F. Kennedy"
  ],
  "answer": [],
  "evidence_paths": [],
  "status": "abstain"
}


#### Comparative Analysis of Two Reasoning Methods

#### 1. Analysis of Method A (Baseline Sub-graph Retrieval)

#### Observations

- **Original Graph**: Method A struggled with the initial context, incorrectly linking the starting entity to "President" rather than the target subject. It failed to navigate the surrounding triples to find the correct reasoning path, resulting in an `abstain` status.
- **Modified Graph (Edge Removed)**: Even when operating on the graph with a corrected starting node ("John F. Kennedy"), Method A absorbed the surrounding 2-hop neighborhood triples but remained unable to deduce the correct answer, maintaining the `abstain` status.

#### Core Analysis

- **Vulnerability to Contextual Noise**
The fundamental flaw of the baseline method lies in its indiscriminate, unguided retrieval. By extracting up to 50 neighboring triples within 2 hops, it injects a massive amount of contextual noise into the prompt. For complex multi-hop queries, the LLM's attention mechanism becomes overwhelmed by irrelevant edges, making it nearly impossible to independently piece together a coherent logical chain.
- **High Dependency on Exact Entity Linking**
Method A's initial failure on the original graph demonstrates that without a structured reasoning framework, the LLM is easily distracted by general nouns in the prompt (e.g., "President"). It lacks the stability to anchor itself to the correct root entity when exposed to a noisy, raw sub-graph.

---

#### 2. Analysis of Method B (RoG-Inspired Pathway Reasoning)

#### Observations

- **Original Graph**: Method B accurately identified "John F. Kennedy" as the root entity, successfully generated the correct relation sequence plan, and extracted the precise 2-hop evidence path (JFK → Vice President → LBJ → President). It successfully outputted the correct answer: "Lyndon B. Johnson".
- **Modified Graph (Edge Removed)**: When the critical bridge triple `['John F. Kennedy', 'government.us_president.vice_president', 'Lyndon B. Johnson']` was manually removed from the graph, Method B's retrieval and grounding module was instantly severed. As a result, the evidence path returned empty `[]`, and the system proactively shifted its status to `abstain`.

#### Core Analysis

- **Demonstration of Faithful Reasoning**
This is the most crucial finding of the edge-removal test. Although the LLM (Gemini) inherently possesses the parametric knowledge to answer the question correctly (it knows LBJ succeeded JFK), Method B effectively suppressed this internal bias. Because the physical supporting edge was missing from the modified knowledge graph, the grounding step (Step 11) yielded an empty set, which strictly forced the final answering step (Step 12) to abstain. This proves that Method B is 100% faithful to the provided graph evidence, successfully mitigating the risk of LLMs generating unsupported, hallucinated answers based on parametric memory.
- **Advantage of Plan-then-Retrieve (Noise Reduction)**
Method B's success on the original graph highlights the immense architectural advantage of explicit relation planning (Step 9). By instructing the LLM to plan the semantic path before interacting with the graph, Method B transforms an undirected, noisy sub-graph search into a highly directional graph traversal. This mechanism perfectly filters out irrelevant neighbor edges, reducing the LLM's cognitive load and dramatically improving multi-hop reasoning accuracy.